# Phase 3 — Spatio-Temporal Feature Engineering
Rainfall-triggered landslide detection pipeline

Scope: extract spectral indices, SAR backscatter/change metrics, rainfall thresholds, and topographic derivatives at each of the 254 points from `phase2_cleaned_dataset.csv`. **Each source is extracted at its own native scale with its own `reduceRegion` call, not one shared scale across a stacked image** — stacking bands of different native resolution into one image and calling a single `reduceRegion` silently reprojects every band to one scale, which was flagged as a real risk earlier in this project.

**Time-relative features (spectral, SAR, rainfall) are computed per-point, each relative to that point's own `event_date`** — not a fixed calendar window — since that's what the model will actually see at inference time. Static topographic features don't depend on date and are batch-extracted once.

## Setup

In [ ]:
import ee
import geemap.core as geemap
import pandas as pd
import numpy as np
import time

ee.Authenticate()
ee.Initialize(project='landslidedetection-506008')  # <-- your GEE-linked Cloud project
print("Earth Engine initialized:", ee.Number(1).add(1).getInfo() == 2)


## Re-declare Phase 1/2 objects + load the cleaned point dataset

In [ ]:
lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

S2_START, S2_END = "2017-01-01", "2023-12-31"

srtm = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(roi)
srtm_slope = ee.Terrain.slope(srtm)
srtm_aspect = ee.Terrain.aspect(srtm)

s2_sr = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).filterDate(S2_START, S2_END))
s2_cloud_prob = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                  .filterBounds(roi).filterDate(S2_START, S2_END))
s2_join = ee.Join.saveFirst('s2cloudless').apply(
    primary=s2_sr, secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index'))
s2_with_prob = ee.ImageCollection(s2_join)

s1_grd = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(roi)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .select(["VV", "VH"]))

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(roi)


def mask_s2_clouds_and_shadows(image, max_cloud_prob=20):
    """Masks clouds (s2cloudless) and cloud shadows (SCL + NIR/SWIR dark-pixel check)."""
    cloud_prob = ee.Image(image.get('s2cloudless')).select('probability')
    is_cloud = cloud_prob.gte(max_cloud_prob)
    scl = image.select('SCL')
    is_shadow_scl = scl.eq(3)
    dark_pixels = image.select('B8').lt(1500).And(image.select('B11').lt(1000))
    shadow_near_cloud = is_cloud.focal_max(radius=60, units='meters').And(dark_pixels)
    combined_mask = is_cloud.Or(is_shadow_scl).Or(shadow_near_cloud)
    optical_bands = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).divide(10000)
    return optical_bands.updateMask(combined_mask.Not()).copyProperties(image, ['system:time_start'])


points_df = pd.read_csv("phase2_cleaned_dataset.csv")
points_df["event_date"] = pd.to_datetime(points_df["event_date"])
print(f"Loaded {len(points_df)} points ({points_df['label'].sum()} positive, "
      f"{(points_df['label']==0).sum()} negative)")
points_df.head()


## 1. Static topographic derivatives
Elevation, slope, aspect, curvature, terrain ruggedness, and TWI — none of these depend on event date, so all 254 points are extracted in a single batched `sampleRegions()` call rather than a per-point loop.

**Two of these are pragmatic approximations, documented as such rather than presented as textbook-exact:**
- **Curvature**: a general Laplacian-kernel second-derivative approximation, not a full plan/profile decomposition (which needs Hessian eigenvalue analysis).
- **TRI (ruggedness)**: elevation standard deviation in a 3×3 neighborhood — a standard, widely-used ruggedness proxy, not the strict Riley et al. TRI formula.
- **TWI**: computed from MERIT Hydro's upstream drainage area band (`MERIT/Hydro/v1_0_1`, band `upa`) combined with slope, following the standard `TWI = ln(upstream_area / tan(slope))` formula.

In [ ]:
curvature = srtm.convolve(ee.Kernel.laplacian8(normalize=False)).rename("curvature")
tri = srtm.reduceNeighborhood(
    reducer=ee.Reducer.stdDev(), kernel=ee.Kernel.square(1)
).rename("tri")

merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")
upstream_area_m2 = merit_hydro.select("upa").multiply(1e6)  # km^2 -> m^2
slope_rad = srtm_slope.multiply(np.pi / 180)
twi = upstream_area_m2.divide(slope_rad.tan().max(1e-6)).log().rename("twi")

topo_image = (srtm.rename("elevation")
              .addBands(srtm_slope.rename("slope"))
              .addBands(srtm_aspect.rename("aspect"))
              .addBands(curvature)
              .addBands(tri)
              .addBands(twi))

points_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {"idx": i})
    for i, row in points_df.iterrows()
])

topo_fc = topo_image.sampleRegions(collection=points_fc, scale=90, geometries=False, tileScale=4)
topo_features = topo_fc.getInfo()["features"]
print(f"Topographic extraction returned {len(topo_features)} / {len(points_df)} points.")

topo_records = [f["properties"] for f in topo_features]
topo_df = pd.DataFrame(topo_records).set_index("idx").sort_index()
print(f"Points missing a topographic reading: {len(points_df) - len(topo_df)}")
topo_df.describe()


## 2. Per-point dynamic feature extraction (spectral, SAR, rainfall)
These genuinely depend on each point's own event date, so they can't be batched the way the topographic features were — each point needs its own time window. Extracted as **one combined `reduceRegion` call per source per point** (3 calls per point: spectral, SAR, rainfall) rather than one call per individual feature, to keep the total number of round trips manageable.

**This will take a while** — roughly 250 points × 3 sources is 750+ server round trips. `TEST_MODE` runs it on 10 points first so a bug surfaces in seconds, not after a 20-minute run.

In [ ]:
TEST_MODE = True  # set False once the 10-point test below looks correct
SAR_WINDOW_DAYS = 12        # pre/post composite window each side of the event
SPECTRAL_WINDOW_STEPS = [30, 60, 90, 120]  # adaptive: try 30 days, widen if no clear pixel
MAX_RETRIES = 4
BASE_BACKOFF_SECONDS = 2


def call_with_retry(fn, *args, **kwargs):
    """Retries a getInfo()-backed call with exponential backoff. The full 254-point run
    showed SAR/rainfall success collapse from 10/10 (small test) to ~4% -- since neither
    source is affected by cloud cover, that pattern is the signature of Earth Engine
    rate-limiting/quota errors under ~2000+ rapid sequential requests, not a real data
    gap. Retrying transient errors, rather than giving up on the first one, is the fix."""
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs), None
        except Exception as e:
            last_error = str(e)[:300]
            time.sleep(BASE_BACKOFF_SECONDS * (2 ** attempt))
    return None, last_error


def get_spectral_features(lon, lat, event_date):
    """Adaptive-window spectral extraction. Monsoon-season cloud cover means a fixed
    30-day pre-event window frequently has zero clear pixels at the point. Widens the
    window until a valid composite is found, and records which window size was needed
    so staler composites are auditable rather than invisible."""
    point = ee.Geometry.Point([lon, lat])
    end = ee.Date(event_date.strftime("%Y-%m-%d"))

    for window in SPECTRAL_WINDOW_STEPS:
        start = end.advance(-window, "day")
        window_coll = s2_with_prob.filterDate(start, end)
        masked = window_coll.map(mask_s2_clouds_and_shadows)
        composite = masked.median()

        ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
        ndwi = composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
        bsi = composite.expression(
            "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
            {"SWIR": composite.select("B11"), "RED": composite.select("B4"),
             "NIR": composite.select("B8"), "BLUE": composite.select("B2")}
        ).rename("BSI")

        img = ndvi.addBands(ndwi).addBands(bsi)
        result, err = call_with_retry(
            lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=10, bestEffort=True).getInfo()
        )
        if err is not None:
            raise RuntimeError(f"window={window}d: {err}")
        if result.get("NDVI") is not None:
            result["spectral_window_days"] = window
            return result

    return {"spectral_window_days": None}


def get_sar_features(lon, lat, event_date):
    point = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date.strftime("%Y-%m-%d"))
    pre = s1_grd.filterDate(event.advance(-SAR_WINDOW_DAYS, "day"), event).mean()
    post = s1_grd.filterDate(event, event.advance(SAR_WINDOW_DAYS, "day")).mean()
    delta = post.subtract(pre)

    img = (pre.rename(["VV_pre", "VH_pre"])
           .addBands(post.rename(["VV_post", "VH_post"]))
           .addBands(delta.rename(["dVV", "dVH"])))
    result, err = call_with_retry(
        lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=10, bestEffort=True).getInfo()
    )
    if err is not None:
        raise RuntimeError(err)
    return result


def get_rainfall_features(lon, lat, event_date, api_decay=0.9):
    point = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date.strftime("%Y-%m-%d"))

    r1 = chirps.filterDate(event.advance(-1, "day"), event).sum().rename("rain_1d")
    r7 = chirps.filterDate(event.advance(-7, "day"), event).sum().rename("rain_7d")
    r30 = chirps.filterDate(event.advance(-30, "day"), event).sum().rename("rain_30d")

    api_window = chirps.filterDate(event.advance(-30, "day"), event)

    def weight(img):
        days_before = event.difference(img.date(), "day")
        return img.multiply(ee.Number(api_decay).pow(days_before))

    api = api_window.map(weight).sum().rename("API")

    img = r1.addBands(r7).addBands(r30).addBands(api)
    result, err = call_with_retry(
        lambda: img.reduceRegion(reducer=ee.Reducer.first(), geometry=point, scale=5566, bestEffort=True).getInfo()
    )
    if err is not None:
        raise RuntimeError(err)
    return result


def extract_all_dynamic(df, verbose_every=10):
    records = []
    n = len(df)
    start_time = time.time()
    for pos, (idx, row) in enumerate(df.iterrows()):
        rec = {"idx": idx}
        try:
            rec.update(get_spectral_features(row.longitude, row.latitude, row.event_date))
        except Exception as e:
            rec["_spectral_error"] = str(e)[:200]
        try:
            rec.update(get_sar_features(row.longitude, row.latitude, row.event_date))
        except Exception as e:
            rec["_sar_error"] = str(e)[:200]
        try:
            rec.update(get_rainfall_features(row.longitude, row.latitude, row.event_date))
        except Exception as e:
            rec["_rainfall_error"] = str(e)[:200]
        records.append(rec)
        if (pos + 1) % verbose_every == 0 or (pos + 1) == n:
            elapsed = time.time() - start_time
            print(f"  {pos+1}/{n} points done ({elapsed:.0f}s elapsed)")
    return pd.DataFrame(records).set_index("idx")


test_subset = points_df.head(10)
print("Running TEST_MODE on 10 points...")
dynamic_test_df = extract_all_dynamic(test_subset)
error_cols = [c for c in dynamic_test_df.columns if c.startswith("_")]
print(f"\nColumns returned: {list(dynamic_test_df.columns)}")
if "spectral_window_days" in dynamic_test_df.columns:
    print(f"\nSpectral window sizes actually used:")
    print(dynamic_test_df["spectral_window_days"].value_counts(dropna=False))
if error_cols:
    print(f"\n⚠️ Errors encountered in test run (actual messages, not just a count):")
    for col in error_cols:
        non_null = dynamic_test_df[col].dropna()
        if len(non_null) > 0:
            print(f"\n{col}:")
            print(non_null)
else:
    print("\n✅ No errors in the 10-point test.")
dynamic_test_df


### Check the test output before scaling up
Look for: no `_..._error` columns populated, and NDVI/NDWI/BSI/rain values in sane ranges (NDVI roughly -1 to 1, rainfall in mm, not all-NaN). If it looks right, set `TEST_MODE = False` above and re-run this cell to process all points.

In [ ]:
if TEST_MODE:
    print("TEST_MODE is still True — set it to False above and re-run to process all points.")
    dynamic_df = dynamic_test_df
else:
    print(f"Running full extraction on {len(points_df)} points — this will take a while...")
    dynamic_df = extract_all_dynamic(points_df)
    error_cols = [c for c in dynamic_df.columns if c.startswith("_")]
    if error_cols:
        n_errors = dynamic_df[error_cols].notna().any(axis=1).sum()
        print(f"\n⚠️ {n_errors} / {len(dynamic_df)} points had at least one extraction error.")
        for col in error_cols:
            non_null = dynamic_df[col].dropna()
            if len(non_null) > 0:
                print(f"\n{col} — {len(non_null)} occurrences, most common messages:")
                print(non_null.value_counts().head(5))
    else:
        print("\n✅ Full extraction completed with no errors.")

dynamic_df.describe()


## 3. Merge static + dynamic features and save

In [ ]:
feature_cols_dynamic = [c for c in dynamic_df.columns if not c.startswith("_")]
error_cols_dynamic = [c for c in dynamic_df.columns if c.startswith("_")]

features = points_df.join(topo_df, how="left").join(dynamic_df[feature_cols_dynamic], how="left")

# Keep error columns in the saved table (not the modeling feature set) so a failed
# extraction is auditable later instead of silently indistinguishable from "genuinely
# no data available."
if error_cols_dynamic:
    features = features.join(dynamic_df[error_cols_dynamic], how="left")

print(f"Final feature table: {len(features)} rows, {len(features.columns)} columns")

all_feature_cols = topo_df.columns.tolist() + feature_cols_dynamic
print("\nMissingness per feature (this is the number that actually matters, not just "
      "'rows missing at least one value' -- a few sparse columns can make nearly every "
      "row look incomplete even if most columns are fine):")
missing_pct = features[all_feature_cols].isna().mean().sort_values(ascending=False) * 100
print(missing_pct.round(1))

if error_cols_dynamic:
    print("\nRows with a recorded extraction error, by source:")
    print(features[error_cols_dynamic].notna().sum())

n_incomplete = features[all_feature_cols].isna().any(axis=1).sum()
print(f"\nRows missing at least one feature value: {n_incomplete} / {len(features)} "
      "(expect this to be high if any single column above has high missingness -- "
      "that's a reason to look at the column-level breakdown above, not a reason to "
      "drop those rows wholesale in the next step).")

features.to_csv("phase3_feature_table.csv", index=False)
print("\nSaved phase3_feature_table.csv")
features.head()


## 4. Multicollinearity check (Pearson correlation + VIF)
Drop or combine features with VIF > 10 before modeling, per the plan — checked here rather than left implicit, so Phase 4 starts from a documented, decorrelated feature set.

In [ ]:
!pip install -q statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor

numeric_feature_cols = topo_df.columns.tolist() + feature_cols_dynamic

# VH_pre/VV_pre are exact linear components of dVH/dVV (dVH = VH_post - VH_pre), so
# keeping both makes the design matrix singular -- that's the actual cause of the
# VIF ~ 1e15 / rank-deficient warnings, not numerical instability. Landslide
# detection cares about *change*, so keep the deltas + post (absolute context)
# and drop the redundant pre-event pair for this multicollinearity check and for
# Phase 4 modeling. VH_pre/VV_pre stay in phase3_feature_table.csv for audit/
# reference -- this only trims the *modeling* feature list.
redundant_sar_cols = ["VH_pre", "VV_pre"]
numeric_feature_cols = [c for c in numeric_feature_cols if c not in redundant_sar_cols]

# Pearson correlation: use pandas' native pairwise-complete handling, NOT a manual
# dropna() across every column at once -- that was the actual bug last run. Any one
# sparse column (spectral features, given monsoon cloud cover) caused almost every
# row to be dropped, collapsing the sample to 1 row and producing meaningless,
# all-NaN correlations and rank-deficient VIF warnings.
corr = features[numeric_feature_cols].corr()  # pairwise-complete by default
print("Pearson correlation matrix (pairwise-complete, not listwise-deleted):")
print(corr.round(2))

# VIF genuinely needs complete cases (it's an OLS regression per feature), so some
# row loss here is unavoidable -- but we report exactly how much, and mean-impute
# for this diagnostic only (never for real modeling) so a few sparse rows don't
# collapse the whole check the way dropna() did.
complete_mask = features[numeric_feature_cols].notna().all(axis=1)
n_complete = complete_mask.sum()
print(f"\nFully complete rows (no imputation): {n_complete} / {len(features)}")

if n_complete >= 30:
    vif_input = features.loc[complete_mask, numeric_feature_cols]
    vif_note = f"using the {n_complete} fully complete rows"
else:
    print(f"Too few complete rows ({n_complete}) for a reliable VIF fit -- "
          "mean-imputing missing values for this diagnostic only.")
    vif_input = features[numeric_feature_cols].fillna(features[numeric_feature_cols].mean())
    vif_note = f"using mean-imputed data ({len(vif_input)} rows) -- diagnostic only, not for modeling"

vif_data = pd.DataFrame({
    "feature": numeric_feature_cols,
    "VIF": [variance_inflation_factor(vif_input.values, i) for i in range(len(numeric_feature_cols))]
}).sort_values("VIF", ascending=False)

print(f"\nVariance Inflation Factors ({vif_note}):")
print(vif_data)

high_vif = vif_data[vif_data["VIF"] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️  {len(high_vif)} feature(s) with VIF > 10 — consider dropping or combining before Phase 4:")
    print(high_vif)
else:
    print("\n✅ No features exceed VIF > 10 — feature set looks reasonably decorrelated.")


---
**Next: Phase 4 — machine learning modeling.** Random Forest + XGBoost, trained on `phase3_feature_table.csv` with **Spatial Cluster K-Fold (KMeans on projected coordinates)** for cross-validation — not the `block_id` grid from Phase 2, since the semivariogram check there showed a fixed grid isn't well-matched to this sample size and ROI.